<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "308"
iso_country = "Grenada"
year = 2024
gdp = 1.350e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Grenada': 

Number of trade records present in dataset: 13783
Number of partner countries present in dataset: 85
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$574.8m
Total weight within the dataset: 785.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,308,Grenada,004,Afghanistan,Import,Sea,901841,901841 | Dental instruments a,9018,9018 | Instruments and appl,90,"90 | Optical, photographi",4813,Other instruments and appliances (except syrin...,Other Manufacturing,41,OME - Machinery and equipment n.e.c.,1683.852,NaN,1683.852,NaN,0.0,NaN,38.055055,15693.0,9751.175103,371.081507,['Container'],['Container'],1.93958,1.936193,1.936524,1.936524,1.938042,1.938042,2.028899,3.595845,2.237941,2.077720,2.083362,2.135542,1.986648,3.521580,2.355574,2.157670,2.108156,2.203634,1.949593,3.829480,2.921024,2.554625,2.363511,2.296840,1.913223,4.071675,3.350521,3.139723,2.530443,2.461637,1.877541,4.126072,3.942097,3.975992,2.743263,2.585871,1.841143,4.041265,3.872770,4.243189,3.138658,2.633266,1.806853,3.604907,3.757272,3.937229,2.951472,2.664379,1.771889,3.311415,3.366140,4.198825,3.468819,2.682023,4813
1,1,308,Grenada,020,Andorra,Import,Sea,970190,970190 | Artwork; collages an,9701,"9701 | Paintings, drawings",97,97 | Works of art; collec,3896,"Paintings, drawings and pastels; original engr...",NaN,42,OMF - Manufactures n.e.c.,49.033,NaN,49.033,NaN,0.0,NaN,0.078453,6875.0,4271.925625,0.335145,['all'],['all'],0.00202,0.002017,0.002017,0.002017,0.002019,0.002019,0.002099,0.003481,0.002255,0.002125,0.002123,0.002187,0.002061,0.003359,0.002351,0.002208,0.002165,0.002245,0.002029,0.003696,0.002854,0.002572,0.002414,0.002325,0.001997,0.003954,0.003252,0.003077,0.002531,0.002468,0.001966,0.003963,0.003805,0.003869,0.002720,0.002574,0.001934,0.003888,0.003731,0.003972,0.003102,0.002614,0.001904,0.003519,0.003613,0.003838,0.002992,0.002640,0.001873,0.003266,0.003315,0.003929,0.003263,0.002655,3896


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $150.2m (26.1%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
02 | Meat and edible meat  21695988.9  10094032.2      3.8      1.3
21 | Miscellaneous edible  18609733.2   5078042.7      3.2      0.6
22 | Beverages, spirits a  17044896.6  74987358.9      3.0      9.6
19 | Preparations of cere  15579501.2   5855840.1      2.7      0.7
04 | Dairy produce; birds  14255979.2   5653240.4      2.5      0.7
20 | Preparations of vege   9900770.6   6403211.3      1.7      0.8
07 | Vegetables and certa   8784262.3  14358186.7      1.5      1.8
10 | Cereals                8089451.8  26049069.5      1.4      3.3
16 | Meat, fish, crustace   8078131.5   1779969.5      1.4      0.2
17 | Sugars and sugar con   5632512.4  12143872.4      1.0      1.5
15 | Animal, vegetable or   4940495.3   3578984.6      0.9      0.5
11 | Products o

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $142.0m (24.7%), ... 

                    cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  78245347.8  107383688.3     13.6     13.7
Agriculture       61213031.5   54460874.1     10.7      6.9
Fishing            2521274.7     466644.1      0.4      0.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $122.7m (21.4%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     55792080.7  26495802.0      9.7      3.4
OMT - Meat products n.e.c.     24639436.0  10690898.4      4.3      1.4
MIL - Dairy products           14703075.0   5464407.5      2.6      0.7
V_F - Vegetables, fruit, nuts  10687680.3  15783825.6      1.9      2.0
VOL - Vegetable oils and fats   6319499.8   7066586.5      1.1      0.9
CMT - Bovine meat prods         3124986.7    763555.3      0.5      0.1
SGR - Sugar                     2609048.2   7932944.3      0.5      1.0
PDR - Paddy rice                1522027.0   2643760.9      0.3      0.3
GRO - Cereal grains n.e.c.      1461008.5   6039729.5      0.3      0.8
OCR - Crops n.e.c.              1146431.8    907108.4      0.2      0.1
PCR - Processed rice             592078.7   1028440.7      0.1      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 26.1% 
GTAP 21.4% 
EORA 24.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16131/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,308,Grenada,004,Afghanistan,Import,Sea,901841,901841 | Dental instruments a,9018,9018 | Instruments and appl,90,"90 | Optical, photographi",4813,Other instruments and appliances (except syrin...,Other Manufacturing,41,OME - Machinery and equipment n.e.c.,1683.852,NaN,1683.852,NaN,0.0,NaN,38.055055,15693.0,9751.175103,371.081507,['Container'],['Container'],1.93958,1.936193,1.936524,1.936524,1.938042,1.938042,2.028899,3.595845,2.237941,2.077720,2.083362,2.135542,1.986648,3.521580,2.355574,2.157670,2.108156,2.203634,1.949593,3.829480,2.921024,2.554625,2.363511,2.296840,1.913223,4.071675,3.350521,3.139723,2.530443,2.461637,1.877541,4.126072,3.942097,3.975992,2.743263,2.585871,1.841143,4.041265,3.872770,4.243189,3.138658,2.633266,1.806853,3.604907,3.757272,3.937229,2.951472,2.664379,1.771889,3.311415,3.366140,4.198825,3.468819,2.682023,4813,NaN
1,1,308,Grenada,020,Andorra,Import,Sea,970190,970190 | Artwork; collages an,9701,"9701 | Paintings, drawings",97,97 | Works of art; collec,3896,"Paintings, drawings and pastels; original engr...",NaN,42,OMF - Manufactures n.e.c.,49.033,NaN,49.033,NaN,0.0,NaN,0.078453,6875.0,4271.925625,0.335145,['all'],['all'],0.00202,0.002017,0.002017,0.002017,0.002019,0.002019,0.002099,0.003481,0.002255,0.002125,0.002123,0.002187,0.002061,0.003359,0.002351,0.002208,0.002165,0.002245,0.002029,0.003696,0.002854,0.002572,0.002414,0.002325,0.001997,0.003954,0.003252,0.003077,0.002531,0.002468,0.001966,0.003963,0.003805,0.003869,0.002720,0.002574,0.001934,0.003888,0.003731,0.003972,0.003102,0.002614,0.001904,0.003519,0.003613,0.003838,0.002992,0.002640,0.001873,0.003266,0.003315,0.003929,0.003263,0.002655,3896,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $102.9m of $574.8m (17.9%)
Volume: 94.1 of 785.0 million kilograms (12.0%)
Value-to-Weight: 1.09 from 0.73 US$ per kg (+49.3%)
Average distance to market (volume-weighted): 2143.5 from 1380.6 nm (+55.3%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    383  40104509.2  22366236.1      7.0      2.8
1        Meat and Prep     54  17370581.1   9062016.6      3.0      1.2
2                Dairy     79  14086201.3   5579106.9      2.5      0.7
3           Vegetables    251  11919531.7  16912321.9      2.1      2.2
4  Unprocessed Cereals     25   5975346.0  22376867.8      1.0      2.9
5      Sugar and Honey     78   5721951.0  12169514.4      1.0      1.6
6        Fats and Oils     90   4347182.8   2916890.6      0.8      0.4
7               Fruits    137   3355006.4   2712817.6      0.6      0.3



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,13095783.6,23126530.9,14424278.3,13402403.5,13436043.5,13776578.8,12127086.0,26525518.8,25349101.9,25577988.4,17665752.6,16655739.5,11450910.2,21318154.0,21669013.1,26957450.2,22276310.0,17269851.7
1,Fats and Oils,3232947.6,5480290.5,3507128.6,3274383.2,3282507.5,3381166.9,3014304.1,6258039.4,5929347.4,6023602.8,4261967.6,4023579.0,2861685.4,5082039.2,5232065.9,6366329.8,5237572.0,4160039.1
2,Fruits,3383571.8,5970623.0,3725662.9,3462338.5,3470866.4,3559051.5,3133748.0,6847524.3,6544276.8,6603959.8,4562036.4,4301411.3,2959364.6,5504948.3,5595374.9,6956899.9,5749207.3,4459716.4
3,Meat and Prep,15784067.7,27974320.1,17410332.6,16163873.7,16207771.1,16613708.5,14606556.8,32099286.1,30668027.4,30931713.5,21341552.7,20117097.9,13784624.0,25761558.6,26187298.2,32665274.8,26986100.4,20865125.0
4,Processed Cereals,17596717.3,30662860.4,19281934.6,17965408.6,18001026.0,18475004.8,16335437.9,35186258.7,33641767.2,34011701.5,23569564.8,22209094.4,15455032.5,28395481.1,28853449.4,35557969.8,29427368.6,23002014.7
5,Sugar and Honey,10084599.2,17142478.0,10953626.1,10229444.4,10255636.5,10551838.7,9399485.9,19722346.4,18737233.2,19038042.2,13371346.3,12565858.6,8921258.4,15970578.2,16351351.6,19908982.2,16454943.9,12991828.8
6,Unprocessed Cereals,26979236.0,47620510.7,29714546.0,27606365.1,27680989.9,28380833.6,24985096.2,54717535.5,52268728.9,52760809.6,36431958.5,34307094.0,23593135.4,43938867.3,44660948.4,55568952.1,45941882.6,35570459.2
7,Vegetables,20381486.2,35949571.4,22436970.4,20853321.6,20904648.2,21436676.3,18878360.3,41235952.6,39407756.8,39772107.1,27472417.1,25903576.4,17828480.3,33150466.7,33696430.4,41875224.5,34605850.5,26855890.5


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.71,0.09,0.02,0.02,0.05,1.02,0.94,0.95,0.39,0.32,0.70,0.73,1.10,0.77,0.41
1,Fats and Oils,0.52,0.06,0.01,0.01,0.03,0.75,0.67,0.69,0.29,0.23,0.51,0.55,0.81,0.55,0.30
2,Fruits,0.77,0.10,0.02,0.03,0.05,1.11,1.02,1.03,0.43,0.35,0.76,0.79,1.19,0.83,0.45
3,Meat and Prep,0.70,0.09,0.02,0.02,0.05,1.01,0.92,0.94,0.39,0.32,0.69,0.71,1.09,0.76,0.41
4,Processed Cereals,0.33,0.04,0.01,0.01,0.02,0.47,0.43,0.44,0.18,0.15,0.32,0.33,0.50,0.35,0.19
5,Sugar and Honey,1.23,0.15,0.03,0.03,0.08,1.80,1.63,1.68,0.69,0.55,1.23,1.30,1.92,1.32,0.71
6,Unprocessed Cereals,3.45,0.46,0.10,0.12,0.23,4.98,4.57,4.65,1.92,1.56,3.40,3.53,5.35,3.74,2.00
7,Vegetables,1.31,0.17,0.04,0.04,0.09,1.88,1.72,1.75,0.72,0.59,1.29,1.33,2.02,1.41,0.76
